In [1]:
from __future__ import annotations

import importlib
import json
import os
import sys
from pathlib import Path
from typing import Any

from mp_api.client import MPRester
from pymatgen.core import Structure

/home/sairam/miniforge3/envs/LightshowAI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def add_repo_to_path(repo_path: str) -> None:
    repo = str(Path(repo_path).resolve())
    if repo not in sys.path:
        sys.path.insert(0, repo)


def fetch_structure(material_id: str, api_key: str) -> Structure:
    if not api_key:
        raise ValueError(
            "No Materials Project API key found. Set MP_API_KEY or assign MP_API_KEY in the config cell."
        )

    with MPRester(api_key) as mpr:
        structure = mpr.get_structure_by_material_id(material_id)

    if structure is None:
        raise RuntimeError(f"No structure returned for material ID: {material_id}")

    return structure


def import_model_module(module_name: str):
    return importlib.import_module(module_name)


def get_callable(module, fn_name: str):
    try:
        fn = getattr(module, fn_name)
    except AttributeError as exc:
        raise AttributeError(
            f"Module '{module.__name__}' does not define function '{fn_name}'"
        ) from exc

    if not callable(fn):
        raise TypeError(
            f"Attribute '{fn_name}' in module '{module.__name__}' is not callable"
        )
    return fn


def structure_summary(structure: Structure) -> dict[str, Any]:
    return {
        "formula": structure.composition.reduced_formula,
        "num_sites": len(structure),
        "lattice": structure.lattice.as_dict(),
        "species": [str(site.specie) for site in structure],
        "cart_coords": [list(map(float, site.coords)) for site in structure],
        "frac_coords": [list(map(float, site.frac_coords)) for site in structure],
    }


def to_jsonable(obj: Any) -> Any:
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    if hasattr(obj, "as_dict"):
        try:
            return obj.as_dict()
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        try:
            return {
                k: to_jsonable(v)
                for k, v in vars(obj).items()
                if not k.startswith("_")
            }
        except Exception:
            pass
    return repr(obj)


def print_block(title: str, obj: Any, pretty: bool = False) -> None:
    print(f"\n=== {title} ===")
    payload = to_jsonable(obj)
    if pretty:
        print(json.dumps(payload, indent=2, sort_keys=False))
    else:
        print(json.dumps(payload))

In [ ]:
MATERIAL_ID = "mp-390" # 
MP_API_KEY = ""  # or paste your key here
PRETTY = True
print(f"Fetching structure for {MATERIAL_ID} from Materials Project...")
structure = fetch_structure(MATERIAL_ID, MP_API_KEY)
print_block("STRUCTURE SUMMARY", structure_summary(structure), pretty=PRETTY)

Fetching structure for mp-390 from Materials Project...


Retrieving MaterialsDoc documents: 100%|██████████| 1/1 [00:00<00:00, 1274.48it/s]


=== STRUCTURE SUMMARY ===
{
  "formula": "TiO2",
  "num_sites": 6,
  "lattice": {
    "@module": "pymatgen.core.lattice",
    "@class": "Lattice",
    "matrix": [
      [
        3.54771631,
        0.0,
        -1.31198863
      ],
      [
        -0.48518786,
        3.51438277,
        -1.31198863
      ],
      [
        0.01802498,
        0.02068314,
        5.50138299
      ]
    ],
    "pbc": [
      true,
      true,
      true
    ]
  },
  "species": [
    "Ti",
    "Ti",
    "O",
    "O",
    "O",
    "O"
  ],
  "cart_coords": [
    [
      2.8055156037500004,
      2.20166001625,
      -0.5926371975000002
    ],
    [
      0.27503782625,
      1.33340589375,
      3.4700429274999998
    ],
    [
      0.8988914949374885,
      2.0493055001875304,
      -0.2956574993344864
    ],
    [
      -0.1003220575625115,
      2.9382427626875303,
      2.4550339956655134
    ],
    [
      3.180875487562512,
      0.5968231473124699,
      0.42237173433448616
    ],
    [
      2.1

In [14]:
import pathlib
import urllib.request
from functools import cache
from typing import List

import numpy as np
import torch
from lightning import LightningModule
from matgl import load_model
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.compute import (
    compute_pair_vector_and_distance,
    compute_theta_and_phi,
    create_line_graph,
)
from matgl.utils.cutoff import polynomial_cutoff
from pymatgen.core import Structure as PymatgenStructure
from torch import nn

# LightshowAI model checkpoints repository:
# https://github.com/AI-multimodal/LightshowAI/tree/main/model_checkpoints

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AI-multimodal/LightshowAI/main/model_checkpoints"

# Notebook-safe project root
PARENT_DIRECTORY = pathlib.Path.cwd().resolve()
MODEL_CHECKPOINTS_PATH = PARENT_DIRECTORY / "model_checkpoints"
XASBLOCKS_PATH = MODEL_CHECKPOINTS_PATH / "xasblock" / "v1.1.1"
M3GNET_PATH = MODEL_CHECKPOINTS_PATH / "M3GNet-MP-2021.2.8-PES"

XASBLOCK_FILES = [
    "Co_FEFF.ckpt",
    "Cr_FEFF.ckpt",
    "Cu_FEFF.ckpt",
    "Cu_VASP.ckpt",
    "Fe_FEFF.ckpt",
    "Mn_FEFF.ckpt",
    "Ni_FEFF.ckpt",
    "Ti_FEFF.ckpt",
    "Ti_VASP.ckpt",
    "V_FEFF.ckpt",
]

M3GNET_FILES = [
    "LICENSE",
    "README.md",
    "model.json",
    "model.pt",
    "state.pt",
]


def _download_file(url: str, destination: pathlib.Path, overwrite: bool = False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        return
    print(f"Downloading {destination.name} ...")
    urllib.request.urlretrieve(url, destination)


def ensure_model_checkpoints(overwrite: bool = False):
    """
    Create model directories and download required checkpoint files
    from the LightshowAI GitHub repository if they are missing.
    """
    XASBLOCKS_PATH.mkdir(parents=True, exist_ok=True)
    M3GNET_PATH.mkdir(parents=True, exist_ok=True)

    for filename in XASBLOCK_FILES:
        url = f"{GITHUB_RAW_BASE}/xasblock/v1.1.1/{filename}"
        dest = XASBLOCKS_PATH / filename
        _download_file(url, dest, overwrite=overwrite)

    for filename in M3GNET_FILES:
        url = f"{GITHUB_RAW_BASE}/M3GNet-MP-2021.2.8-PES/{filename}"
        dest = M3GNET_PATH / filename
        _download_file(url, dest, overwrite=overwrite)


# Ensure checkpoints are present before any model code runs
ensure_model_checkpoints()

AVAILABLE_COMBINATIONS = sorted(f.stem for f in XASBLOCKS_PATH.glob("*.ckpt"))


class XASBlock(nn.Sequential):
    DROPOUT = 0.5

    def __init__(self, input_dim: int, hidden_dims: List[int], output_dim: int):
        dims = [input_dim] + hidden_dims + [output_dim]
        layers = []
        for i, (w1, w2) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(w1, w2))
            if i < len(dims) - 2:
                layers.append(nn.BatchNorm1d(w2))
                layers.append(nn.SiLU())
                layers.append(nn.Dropout(self.DROPOUT))
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


class XASBlockModule(LightningModule):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    @classmethod
    def load(
        cls,
        element: str,
        spectroscopy_type: str,
        pattern=XASBLOCKS_PATH / "{element}_{type}.ckpt",
    ):
        ensure_model_checkpoints()
        pattern = str(pattern)
        path = pattern.format(element=element, type=spectroscopy_type)

        if not pathlib.Path(path).exists():
            raise FileNotFoundError(
                f"Checkpoint not found: {path}\n"
                f"Available combinations: {AVAILABLE_COMBINATIONS}"
            )

        model = XASBlock(
            input_dim=64,
            hidden_dims=[500, 500, 550],
            output_dim=141,
        )
        module = cls.load_from_checkpoint(checkpoint_path=path, model=model)
        return module


class M3GNetFeaturizer:
    def __init__(self, model=None, n_blocks=None):
        self.model = model or M3GNetFeaturizer._load_m3gnet()
        self.model.eval()
        self.n_blocks = n_blocks or self.model.n_blocks

    def featurize(
        self,
        structure: PymatgenStructure,
    ):
        graph_converter = Structure2Graph(
            self.model.element_types, self.model.cutoff
        )
        g, state_attr = graph_converter.get_graph(structure)

        node_types = g.ndata["node_type"]
        bond_vec, bond_dist = compute_pair_vector_and_distance(g)

        g.edata["bond_vec"] = bond_vec.to(g.device)
        g.edata["bond_dist"] = bond_dist.to(g.device)

        with torch.no_grad():
            expanded_dists = self.model.bond_expansion(g.edata["bond_dist"])

            l_g = create_line_graph(g, self.model.threebody_cutoff)

            l_g.apply_edges(compute_theta_and_phi)
            g.edata["rbf"] = expanded_dists
            three_body_basis = self.model.basis_expansion(l_g)
            three_body_cutoff = polynomial_cutoff(
                g.edata["bond_dist"], self.model.threebody_cutoff
            )
            node_feat, edge_feat, state_feat = self.model.embedding(
                node_types, g.edata["rbf"], state_attr
            )

            for i in range(self.n_blocks):
                edge_feat = self.model.three_body_interactions[i](
                    g,
                    l_g,
                    three_body_basis,
                    three_body_cutoff,
                    node_feat,
                    edge_feat,
                )
                edge_feat, node_feat, state_feat = self.model.graph_layers[i](
                    g, edge_feat, node_feat, state_feat
                )

        res = np.array(node_feat.detach().cpu().numpy())
        return res

    @cache
    @staticmethod
    def _load_m3gnet(path=M3GNET_PATH):
        ensure_model_checkpoints()
        model = load_model(path).model
        model.eval()
        return model


class XASModel:
    featurizer = M3GNetFeaturizer()

    def __init__(self, element: str, spectroscopy_type: str):
        self.element = element
        self.spectroscopy_type = spectroscopy_type
        self.model = XASBlockModule.load(
            element=element, spectroscopy_type=spectroscopy_type
        )
        self.model.eval()

    def _get_feature(self, structure: PymatgenStructure):
        return self.featurizer.featurize(structure)

    def predict(
        self,
        structure: PymatgenStructure,
    ):
        with torch.no_grad():
            feature = self._get_feature(structure)
            feature = feature * 1000.0
            device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
            feature = torch.tensor(feature, device=device)
            spectrum = self.model(feature)

        spectrum = spectrum.detach().cpu().numpy().squeeze()
        return spectrum


def predict(structure, absorbing_site, spectroscopy_type):
    site_idxs = [
        ii
        for ii, site in enumerate(structure.sites)
        if site.specie.symbol == absorbing_site
    ]
    if len(site_idxs) == 0:
        raise ValueError(
            f"element {absorbing_site} not found in provided structure"
        )

    spec = XASModel(
        element=absorbing_site, spectroscopy_type=spectroscopy_type
    ).predict(structure)

    result = {ii: spec[ii] for ii in site_idxs}
    return result

In [ ]:
element = "Ti"
theory = "VASP"
output = predict(structure, element, theory)
print(output)

{0: array([0.02362889, 0.02362992, 0.02363205, 0.02363587, 0.02364374,
       0.0236784 , 0.02372479, 0.02378211, 0.02383053, 0.02385101,
       0.02385839, 0.02391419, 0.02419167, 0.02491961, 0.02619175,
       0.02765366, 0.02903827, 0.02987733, 0.03069442, 0.0331122 ,
       0.03772895, 0.0441726 , 0.05351771, 0.06790829, 0.08948899,
       0.12060843, 0.1546817 , 0.17609945, 0.19239151, 0.2134919 ,
       0.23731533, 0.25572243, 0.2620323 , 0.2629814 , 0.27234355,
       0.28267223, 0.27829903, 0.264795  , 0.2566204 , 0.25712892,
       0.26029456, 0.2606097 , 0.25488234, 0.2401852 , 0.21689832,
       0.19300379, 0.175157  , 0.16493124, 0.16103409, 0.16204946,
       0.16755633, 0.17870279, 0.19360235, 0.20971845, 0.2276461 ,
       0.24893399, 0.27324548, 0.3010008 , 0.33265075, 0.36526546,
       0.39628288, 0.42684197, 0.46245596, 0.5034264 , 0.5463719 ,
       0.5894816 , 0.63235164, 0.67480004, 0.7175982 , 0.76099885,
       0.80164236, 0.83833414, 0.8775754 , 0.9299111 , 0.9